<a href="https://colab.research.google.com/github/anhhao73/Tr-tu-nh-n-t-o/blob/main/BTVN_b2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scikit-fuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 23.1 MB/s eta 0:00:00


In [4]:
# BÀI 2.11: HỆ THỐNG GIÁ TIỀN & ĐIỂM THƯỞNG
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

khoang_cach = ctrl.Antecedent(np.arange(0, 51, 1), 'khoang_cach')      # km (0 - 50)
giao_thong = ctrl.Antecedent(np.arange(0, 101, 1), 'giao_thong')       # % kẹt xe (0 - 100)
nhu_cau = ctrl.Antecedent(np.arange(0, 101, 1), 'nhu_cau')             # % mức cầu (0 - 100)
thoi_tiet = ctrl.Antecedent(np.arange(0, 11, 1), 'thoi_tiet')          # Thang điểm thời tiết (0: Xấu nhất, 10: Tốt nhất)
danh_gia_kh = ctrl.Antecedent(np.arange(1.0, 5.1, 0.1), 'danh_gia_kh') # Sao đánh giá (1.0 - 5.0)
dung_gio = ctrl.Antecedent(np.arange(0, 101, 1), 'dung_gio')           # % Đúng giờ (0 - 100)
# 2. KHAI BÁO BIẾN ĐẦU RA (Outputs)
gia_xe = ctrl.Consequent(np.arange(0, 101, 1), 'gia_xe')               # % Mức giá
diem_thuong = ctrl.Consequent(np.arange(0, 101, 1), 'diem_thuong')     # % Điểm thưởng
# 3. THIẾT LẬP HÀM LIÊN THUỘC (Membership Functions)
# Khoảng cách
khoang_cach['ngan'] = fuzz.trimf(khoang_cach.universe, [0, 0, 3])
khoang_cach['trung_binh'] = fuzz.trimf(khoang_cach.universe, [2, 5, 8])
khoang_cach['dai'] = fuzz.trimf(khoang_cach.universe, [6, 13, 20])
khoang_cach['rat_xa'] = fuzz.trapmf(khoang_cach.universe, [15, 25, 50, 50])

# Giao thông, Nhu cầu, Thời tiết
giao_thong['thap'] = fuzz.trimf(giao_thong.universe, [0, 0, 30])
giao_thong['trung_binh'] = fuzz.trimf(giao_thong.universe, [20, 45, 70])
giao_thong['cao'] = fuzz.trapmf(giao_thong.universe, [60, 80, 100, 100])

nhu_cau['thap'] = fuzz.trimf(nhu_cau.universe, [0, 0, 30])
nhu_cau['trung_binh'] = fuzz.trimf(nhu_cau.universe, [20, 45, 70])
nhu_cau['cao'] = fuzz.trapmf(nhu_cau.universe, [60, 80, 100, 100])

thoi_tiet['xau'] = fuzz.trimf(thoi_tiet.universe, [0, 0, 4])
thoi_tiet['trung_binh'] = fuzz.trimf(thoi_tiet.universe, [3, 5, 7]) # Sách ghi: Trung bình/Vừa phải
thoi_tiet['tot'] = fuzz.trimf(thoi_tiet.universe, [6, 10, 10])

# Đánh giá & Đúng giờ
danh_gia_kh['kem'] = fuzz.trimf(danh_gia_kh.universe, [1.0, 1.0, 2.5])
danh_gia_kh['trung_binh'] = fuzz.trimf(danh_gia_kh.universe, [2.0, 3.0, 4.0])
danh_gia_kh['tot'] = fuzz.trapmf(danh_gia_kh.universe, [3.5, 4.5, 5.0, 5.0])

dung_gio['tre'] = fuzz.trimf(dung_gio.universe, [0, 0, 50])
dung_gio['dung_gio'] = fuzz.trimf(dung_gio.universe, [40, 60, 80])
dung_gio['som'] = fuzz.trapmf(dung_gio.universe, [70, 85, 100, 100])

# Đầu ra: Giá xe & Điểm thưởng
gia_xe.automf(names=['thap', 'trung_binh', 'cao', 'rat_cao'])
diem_thuong.automf(names=['khong_co', 'it', 'trung_binh', 'cao'])
# 4. THIẾT LẬP 20 LUẬT MỜ
# --- PHẦN 1: 10 LUẬT ĐỊNH GIÁ CƯỚC ---
r1 = ctrl.Rule(khoang_cach['ngan'] & giao_thong['thap'] & nhu_cau['thap'], gia_xe['thap'])
r2 = ctrl.Rule(khoang_cach['ngan'] & giao_thong['trung_binh'] & nhu_cau['cao'], gia_xe['trung_binh'])
r3 = ctrl.Rule(khoang_cach['trung_binh'] & giao_thong['cao'] & nhu_cau['cao'], gia_xe['cao'])
r4 = ctrl.Rule(khoang_cach['dai'] & giao_thong['trung_binh'] & thoi_tiet['tot'], gia_xe['trung_binh'])
r5 = ctrl.Rule(khoang_cach['dai'] & giao_thong['cao'] & thoi_tiet['xau'], gia_xe['rat_cao'])
r6 = ctrl.Rule(khoang_cach['rat_xa'] & giao_thong['cao'] & nhu_cau['cao'], gia_xe['rat_cao'])
r7 = ctrl.Rule(khoang_cach['trung_binh'] & giao_thong['thap'] & nhu_cau['thap'], gia_xe['trung_binh'])
r8 = ctrl.Rule(khoang_cach['ngan'] & giao_thong['cao'] & thoi_tiet['xau'], gia_xe['cao'])
r9 = ctrl.Rule(khoang_cach['rat_xa'] & thoi_tiet['xau'], gia_xe['rat_cao'])
r10 = ctrl.Rule(khoang_cach['trung_binh'] & giao_thong['trung_binh'] & thoi_tiet['trung_binh'], gia_xe['trung_binh'])

# --- PHẦN 2: 10 LUẬT TÍNH ĐIỂM THƯỞNG ---
r11 = ctrl.Rule(danh_gia_kh['tot'] & dung_gio['som'], diem_thuong['cao'])
r12 = ctrl.Rule(danh_gia_kh['trung_binh'] & dung_gio['dung_gio'], diem_thuong['trung_binh'])
r13 = ctrl.Rule(danh_gia_kh['kem'] & dung_gio['tre'], diem_thuong['khong_co'])
r14 = ctrl.Rule(khoang_cach['dai'] & giao_thong['cao'] & dung_gio['dung_gio'], diem_thuong['cao'])
r15 = ctrl.Rule(khoang_cach['trung_binh'] & giao_thong['trung_binh'] & danh_gia_kh['tot'], diem_thuong['trung_binh'])
r16 = ctrl.Rule(danh_gia_kh['kem'] & dung_gio['tre'], diem_thuong['khong_co'])
r17 = ctrl.Rule(khoang_cach['rat_xa'] & thoi_tiet['xau'] & danh_gia_kh['tot'], diem_thuong['cao'])
r18 = ctrl.Rule(khoang_cach['ngan'] & danh_gia_kh['trung_binh'] & dung_gio['dung_gio'], diem_thuong['it'])
r19 = ctrl.Rule(khoang_cach['dai'] & giao_thong['cao'] & dung_gio['tre'], diem_thuong['it'])
r20 = ctrl.Rule(khoang_cach['trung_binh'] & thoi_tiet['trung_binh'] & danh_gia_kh['tot'], diem_thuong['trung_binh'])

# 5. TẠO HỆ THỐNG VÀ MÔ PHỎNG

grab_ctrl = ctrl.ControlSystem([r1, r2, r3, r4, r5, r6, r7, r8, r9, r10,
                                r11, r12, r13, r14, r15, r16, r17, r18, r19, r20])
grab_sim = ctrl.ControlSystemSimulation(grab_ctrl)

grab_sim.input['khoang_cach'] = 18
grab_sim.input['giao_thong'] = 85
grab_sim.input['nhu_cau'] = 80
grab_sim.input['thoi_tiet'] = 2
grab_sim.input['danh_gia_kh'] = 4.8
grab_sim.input['dung_gio'] = 60

# Thực thi tính toán
grab_sim.compute()

# In kết quả
print("KẾT QUẢ ĐỊNH GIÁ & TÍNH ĐIỂM GRAB-BIKE:")
print("-" * 40)
print(f"-> Mức định giá cước: {grab_sim.output['gia_xe']:.2f}% (Mức độ tăng giá)")
print(f"-> Điểm thưởng cho tài xế: {grab_sim.output['diem_thuong']:.2f}% (Tỷ lệ nhận thưởng tối đa)")

KẾT QUẢ ĐỊNH GIÁ & TÍNH ĐIỂM GRAB-BIKE:
----------------------------------------
-> Mức định giá cước: 85.68% (Mức độ tăng giá)
-> Điểm thưởng cho tài xế: 85.68% (Tỷ lệ nhận thưởng tối đa)


In [18]:
# BÀI 2.12:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

print("=" * 60)
print(" HỆ THỐNG AI QUẢN TRỊ CHIẾT KHẤU SHOPEE (BÀI 2.12 FULL)")
print("=" * 60)

# 1. Khai báo Biến đầu vào (Inputs)
xep_hang = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'xep_hang')           # 0 - 5.0 Sao
khoi_luong = ctrl.Antecedent(np.arange(0, 101, 1), 'khoi_luong')         # 0 - 100%
bien_loi_nhuan = ctrl.Antecedent(np.arange(0, 101, 1), 'bien_loi_nhuan') # 0 - 100%
su_kien = ctrl.Antecedent(np.arange(0, 101, 1), 'su_kien')               # 0 - 100%
doi_thu = ctrl.Antecedent(np.arange(0, 101, 1), 'doi_thu')               # 0 - 100%

# Khai báo Biến đầu ra (Output)
chiet_khau = ctrl.Consequent(np.arange(0, 71, 1), 'chiet_khau')          # Max 70%

# 2. Thiết lập Hàm liên thuộc (Membership Functions)
xep_hang['thap'] = fuzz.trapmf(xep_hang.universe, [0, 0, 3.8, 4.0])
xep_hang['trung_binh'] = fuzz.trimf(xep_hang.universe, [3.9, 4.25, 4.6])
xep_hang['cao'] = fuzz.trapmf(xep_hang.universe, [4.5, 4.8, 5.0, 5.0])

khoi_luong.automf(names=['thap', 'trung_binh', 'cao'])
bien_loi_nhuan.automf(names=['thap', 'trung_binh', 'cao'])
su_kien.automf(names=['khong_co', 'trung_binh', 'cao']) # Từ 'khong_co' khớp với mô tả trang 90
doi_thu.automf(names=['thap', 'trung_binh', 'cao'])

# Thiết lập chính xác các mốc % chiết khấu theo giáo trình
chiet_khau['rat_thap'] = fuzz.trapmf(chiet_khau.universe, [0, 0, 2.5, 5])
chiet_khau['thap'] = fuzz.trimf(chiet_khau.universe, [4, 7.5, 11])
chiet_khau['trung_binh'] = fuzz.trimf(chiet_khau.universe, [9, 15, 21])
chiet_khau['cao'] = fuzz.trimf(chiet_khau.universe, [19, 30, 41])
chiet_khau['rat_cao'] = fuzz.trapmf(chiet_khau.universe, [39, 55, 70, 70])

# 3. Khai báo 7 Luật mờ (Rules)
# - 5 Luật gốc từ trang 89:
rule1 = ctrl.Rule(xep_hang['cao'] & khoi_luong['cao'] & bien_loi_nhuan['cao'], chiet_khau['rat_thap'])
rule2 = ctrl.Rule(xep_hang['thap'] & khoi_luong['thap'] & bien_loi_nhuan['cao'], chiet_khau['cao'])
rule3 = ctrl.Rule(su_kien['cao'] & doi_thu['cao'], chiet_khau['rat_cao'])
rule4 = ctrl.Rule(xep_hang['trung_binh'] & khoi_luong['trung_binh'] & bien_loi_nhuan['trung_binh'], chiet_khau['trung_binh'])
rule5 = ctrl.Rule(doi_thu['thap'] & bien_loi_nhuan['thap'] & khoi_luong['cao'], chiet_khau['rat_thap'])
# - 2 Luật bổ sung từ trang 90:
rule6 = ctrl.Rule(xep_hang['thap'] & su_kien['khong_co'], chiet_khau['trung_binh'])
rule7 = ctrl.Rule(khoi_luong['thap'] & bien_loi_nhuan['thap'], chiet_khau['rat_cao'])

# Khởi tạo mô phỏng
shopee_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7])
shopee_sim = ctrl.ControlSystemSimulation(shopee_ctrl)

# PHẦN 2: HÀM TÍNH TOÁN TÀI CHÍNH (Dựa trên TRANG 88)
def phan_tich_tai_chinh(gia_goc, chi_phi, phan_tram_giam):
    gia_ban_moi = gia_goc * (1 - phan_tram_giam / 100)
    loi_nhuan = gia_ban_moi - chi_phi
    ty_suat_loi_nhuan = (loi_nhuan / gia_ban_moi) * 100 if gia_ban_moi > 0 else 0
    return gia_ban_moi, loi_nhuan, ty_suat_loi_nhuan

# PHẦN 3: THỰC THI TEST CASE (Dữ liệu từ TRANG 88 & 90)

# Dữ liệu sản phẩm (Trang 88)
gia_niem_yet = 100000  # 100k
chi_phi_sp = 70000     # 70k

# Cập nhật môi trường hiện tại của cửa hàng (Trang 90)
shopee_sim.input['xep_hang'] = 4.3         # Trung bình
shopee_sim.input['khoi_luong'] = 50        # Trung bình (50%)
shopee_sim.input['bien_loi_nhuan'] = 20    # Thấp (20%)
shopee_sim.input['su_kien'] = 85           # Cao (Vào mùa sale)
shopee_sim.input['doi_thu'] = 80           # Cao (Đối thủ đang xả hàng)

# 1. AI Tính toán % chiết khấu
shopee_sim.compute()
ai_de_xuat_giam_gia = shopee_sim.output['chiet_khau']

# 2. Đưa % chiết khấu vào máy tính tài chính
gia_sau_giam, tien_lai, ty_suat = phan_tich_tai_chinh(gia_niem_yet, chi_phi_sp, ai_de_xuat_giam_gia)

# IN BÁO CÁO KẾT QUẢ ĐẦU RA

print("\n[THÔNG SỐ ĐẦU VÀO TỪ CỬA HÀNG]")
print(f"- Giá gốc: {gia_niem_yet:,.0f} đ | Chi phí: {chi_phi_sp:,.0f} đ")

print("\n[1] AI ĐỀ XUẤT CHIẾT KHẤU")
print(f"-> Mức giảm giá hệ thống tính toán: {ai_de_xuat_giam_gia:.2f}%")
if 30 <= ai_de_xuat_giam_gia <= 45:
    print("-> Nằm trong vùng CAO: 30-40% do sự kiện & đối thủ.")

print("\n[2] PHÂN TÍCH HIỆU QUẢ TÀI CHÍNH")
print(f"- Giá bán thực tế khách trả: {gia_sau_giam:,.0f} đ")
print(f"- Tiền lãi mang về: {tien_lai:,.0f} đ")
print(f"- Tỷ suất lợi nhuận (Biên LN mới): {ty_suat:.2f}%")

print("\n[KẾT LUẬN QUẢN TRỊ]")
if ty_suat > 0:
    print("-> Cửa hàng vẫn có LÃI để cạnh tranh trong mùa sale.")
    if ty_suat < 10:
        print("-> CẢNH BÁO: Biên lợi nhuận đang rất mỏng. Nên cân nhắc dùng 'Chiết khấu bậc thang' hoặc 'Voucher hoàn xu' để tránh rủi ro thay vì giảm giá trực tiếp.")
else:
    print("-> BÁO ĐỘNG ĐỎ: Với mức giảm giá này, cửa hàng đang LỖ. Cần giữ giá hoặc rút khỏi cuộc chiến giá cả.")
print("=" * 60)

 HỆ THỐNG AI QUẢN TRỊ CHIẾT KHẤU SHOPEE (BÀI 2.12 FULL)

[THÔNG SỐ ĐẦU VÀO TỪ CỬA HÀNG]
- Giá gốc: 100,000 đ | Chi phí: 70,000 đ

[1] AI ĐỀ XUẤT CHIẾT KHẤU
-> Mức giảm giá hệ thống tính toán: 48.56%

[2] PHÂN TÍCH HIỆU QUẢ TÀI CHÍNH
- Giá bán thực tế khách trả: 51,444 đ
- Tiền lãi mang về: -18,556 đ
- Tỷ suất lợi nhuận (Biên LN mới): -36.07%

[KẾT LUẬN QUẢN TRỊ]
-> BÁO ĐỘNG ĐỎ: Với mức giảm giá này, cửa hàng đang LỖ. Cần giữ giá hoặc rút khỏi cuộc chiến giá cả.


In [16]:
# BÀI 2.13:

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

print("=" * 65)
print(" HỆ THỐNG LOGIC MỜ: CHIẾN LƯỢC BÁN HÀNG NGÁCH (BÀI 2.13)")
print("=" * 65)

# 1. KHAI BÁO CÁC BIẾN ĐẦU VÀO VÀ ĐẦU RA

nhu_cau_sp = ctrl.Antecedent(np.arange(0, 101, 1), 'nhu_cau_sp')
ap_luc_gia = ctrl.Antecedent(np.arange(0, 101, 1), 'ap_luc_gia')
uy_tin = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'uy_tin')
bien_loi_nhuan = ctrl.Antecedent(np.arange(0, 101, 1), 'bien_loi_nhuan')
nhu_cau_mua = ctrl.Antecedent(np.arange(0, 101, 1), 'nhu_cau_mua')

chiet_khau = ctrl.Consequent(np.arange(0, 71, 1), 'chiet_khau')

# 2. THIẾT LẬP HÀM LIÊN THUỘC (Membership Functions)

uy_tin['thap'] = fuzz.trapmf(uy_tin.universe, [0, 0, 3.8, 4.0])
uy_tin['trung_binh'] = fuzz.trimf(uy_tin.universe, [3.9, 4.25, 4.6])
uy_tin['cao'] = fuzz.trapmf(uy_tin.universe, [4.5, 4.8, 5.0, 5.0])

nhu_cau_sp.automf(names=['thap', 'trung_binh', 'cao'])
ap_luc_gia.automf(names=['thap', 'trung_binh', 'cao'])
bien_loi_nhuan.automf(names=['thap', 'trung_binh', 'cao'])
nhu_cau_mua.automf(names=['khong_co', 'trung_binh', 'cao'])

chiet_khau['rat_thap'] = fuzz.trapmf(chiet_khau.universe, [0, 0, 2.5, 5])
chiet_khau['thap'] = fuzz.trimf(chiet_khau.universe, [4, 7.5, 11])
chiet_khau['trung_binh'] = fuzz.trimf(chiet_khau.universe, [9, 15, 21])
chiet_khau['cao'] = fuzz.trimf(chiet_khau.universe, [19, 30, 41])
chiet_khau['rat_cao'] = fuzz.trapmf(chiet_khau.universe, [39, 55, 70, 70])

# 3. KHAI BÁO 7 LUẬT MỜ

rule1 = ctrl.Rule(nhu_cau_sp['cao'] & ap_luc_gia['thap'] & bien_loi_nhuan['thap'], chiet_khau['rat_thap'])
rule2 = ctrl.Rule(nhu_cau_sp['thap'] & ap_luc_gia['cao'] & bien_loi_nhuan['cao'], chiet_khau['cao'])
rule3 = ctrl.Rule(uy_tin['cao'] & bien_loi_nhuan['trung_binh'] & nhu_cau_mua['cao'], chiet_khau['trung_binh'])
rule4 = ctrl.Rule(ap_luc_gia['cao'] & nhu_cau_mua['cao'] & bien_loi_nhuan['cao'], chiet_khau['rat_cao'])
rule5 = ctrl.Rule(uy_tin['thap'] & nhu_cau_sp['trung_binh'] & bien_loi_nhuan['thap'], chiet_khau['trung_binh'])
rule6 = ctrl.Rule(nhu_cau_sp['cao'] & nhu_cau_mua['khong_co'] & ap_luc_gia['thap'], chiet_khau['rat_thap'])
rule7 = ctrl.Rule(bien_loi_nhuan['cao'] & ap_luc_gia['trung_binh'] & nhu_cau_mua['trung_binh'], chiet_khau['trung_binh'])

ngach_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7])
ngach_sim = ctrl.ControlSystemSimulation(ngach_ctrl)

# 4. CHẠY TÌNH HUỐNG THỰC TẾ (Sản phẩm: Đồng hồ xa xỉ thủ công)

ngach_sim.input['nhu_cau_sp'] = 85
ngach_sim.input['ap_luc_gia'] = 50
ngach_sim.input['uy_tin'] = 4.2
ngach_sim.input['bien_loi_nhuan'] = 85
ngach_sim.input['nhu_cau_mua'] = 85

ngach_sim.compute()
ket_qua_chiet_khau = ngach_sim.output['chiet_khau']

# 5. IN BÁO CÁO

print("\n[TÌNH HUỐNG]: ĐỒNG HỒ XA XỈ THỦ CÔNG - MÙA SALE LỚN")

print("-" * 65)

print(f"-> GỢI Ý CHIẾT KHẤU TỪ AI: {ket_qua_chiet_khau:.2f}%\n")

if 10 <= ket_qua_chiet_khau <= 25:
    print("-> ĐÁNH GIÁ: KẾT QUẢ CHÍNH XÁC THEO LÝ THUYẾT.\n")
    print("Giải thích logic của hệ thống:")
    print("1. Thuật toán mờ đã điều chỉnh mức giảm giá về quanh khoảng\n"
          "   'Trung bình' (15-25%) đúng như dự đoán trong giáo trình.\n")
    print("2. Dù đang trong mùa Sale (Nhu cầu mua cao) nhưng vì đây là\n"
          "   sản phẩm ngách, xa xỉ (Áp lực định giá từ đối thủ chỉ ở\n"
          "   mức Trung bình), nên hệ thống AI đã tự động hãm mức\n"
          "   chiết khấu lại.\n")
    print("3. Việc này giúp cửa hàng kích cầu thành công nhưng vẫn\n"
          "   không làm phá giá hay làm mất đi định vị cao cấp của\n"
          "   thương hiệu.")
else:
    print("-> Kết quả nằm ngoài vùng kỳ vọng Trung bình. Cần kiểm tra\n"
          "   lại các tham số đầu vào.")
print("=" * 65)

 HỆ THỐNG LOGIC MỜ: CHIẾN LƯỢC BÁN HÀNG NGÁCH (BÀI 2.13)

[TÌNH HUỐNG]: ĐỒNG HỒ XA XỈ THỦ CÔNG - MÙA SALE LỚN
-----------------------------------------------------------------
-> GỢI Ý CHIẾT KHẤU TỪ AI: 15.00%

-> ĐÁNH GIÁ: KẾT QUẢ CHÍNH XÁC THEO LÝ THUYẾT.

Giải thích logic của hệ thống:
1. Thuật toán mờ đã điều chỉnh mức giảm giá về quanh khoảng
   'Trung bình' (15-25%) đúng như dự đoán trong giáo trình.

2. Dù đang trong mùa Sale (Nhu cầu mua cao) nhưng vì đây là
   sản phẩm ngách, xa xỉ (Áp lực định giá từ đối thủ chỉ ở
   mức Trung bình), nên hệ thống AI đã tự động hãm mức
   chiết khấu lại.

3. Việc này giúp cửa hàng kích cầu thành công nhưng vẫn
   không làm phá giá hay làm mất đi định vị cao cấp của
   thương hiệu.


In [17]:
# BÀI 2.14
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

print("=" * 65)
print(" HỆ THỐNG ĐIỀU PHỐI VÀ TỐI ƯU GIAO NHẬN LOGISTICS (BÀI 2.14)")
print("=" * 65)
mat_do = ctrl.Antecedent(np.arange(0, 101, 1), 'mat_do')
khan_cap = ctrl.Antecedent(np.arange(0, 101, 1), 'khan_cap')
tai_trong = ctrl.Antecedent(np.arange(0, 101, 1), 'tai_trong')
giao_thong = ctrl.Antecedent(np.arange(0, 101, 1), 'giao_thong')
loi_nhuan = ctrl.Antecedent(np.arange(0, 101, 1), 'loi_nhuan')

ket_hop = ctrl.Consequent(np.arange(0, 101, 1), 'ket_hop')
uu_tien = ctrl.Consequent(np.arange(0, 101, 1), 'uu_tien')
# Tự định nghĩa các hàm tam giác cắt nhau (Overlap) cho toàn bộ đầu vào
for var in [mat_do, khan_cap, tai_trong, giao_thong, loi_nhuan]:
    var['thap'] = fuzz.trimf(var.universe, [0, 0, 60])        # Kéo đuôi 'Thấp' dài đến 60
    var['trung_binh'] = fuzz.trimf(var.universe, [20, 50, 80])
    var['cao'] = fuzz.trimf(var.universe, [40, 100, 100])     # Kéo chân 'Cao' lui về 40

# Định nghĩa cho đầu ra Kết hợp đơn hàng
ket_hop['it'] = fuzz.trimf(ket_hop.universe, [0, 0, 50])
ket_hop['mot_so'] = fuzz.trimf(ket_hop.universe, [20, 50, 80])
ket_hop['nhieu'] = fuzz.trimf(ket_hop.universe, [50, 100, 100])

# Định nghĩa cho đầu ra Ưu tiên
uu_tien['thap'] = fuzz.trimf(uu_tien.universe, [0, 0, 50])
uu_tien['trung_binh'] = fuzz.trimf(uu_tien.universe, [20, 50, 80])
uu_tien['cao'] = fuzz.trimf(uu_tien.universe, [50, 100, 100])

# 5 Luật Kết hợp
rule1 = ctrl.Rule(mat_do['cao'] & tai_trong['thap'] & giao_thong['thap'], ket_hop['nhieu'])
rule2 = ctrl.Rule(mat_do['trung_binh'] & giao_thong['cao'] & khan_cap['trung_binh'], ket_hop['mot_so'])
rule3 = ctrl.Rule(tai_trong['cao'] & mat_do['cao'] & loi_nhuan['trung_binh'], ket_hop['mot_so'])
rule4 = ctrl.Rule(mat_do['thap'] & khan_cap['cao'] & giao_thong['trung_binh'], ket_hop['mot_so'])
rule5 = ctrl.Rule(loi_nhuan['cao'] & khan_cap['cao'] & giao_thong['cao'], ket_hop['mot_so'])

# 3 Luật Ưu tiên
rule6 = ctrl.Rule(khan_cap['cao'] & loi_nhuan['cao'], uu_tien['cao'])
rule7 = ctrl.Rule(khan_cap['trung_binh'] & giao_thong['trung_binh'], uu_tien['trung_binh'])
rule8 = ctrl.Rule(khan_cap['thap'] & mat_do['cao'] & loi_nhuan['thap'], uu_tien['thap'])

giao_nhan_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8])
gn_sim = ctrl.ControlSystemSimulation(giao_nhan_ctrl)

gn_sim.input['mat_do'] = 85        # Cao
gn_sim.input['khan_cap'] = 50      # Trung bình
gn_sim.input['tai_trong'] = 15     # Thấp
gn_sim.input['giao_thong'] = 50    # Trung bình (Nhờ mở rộng MFs, biến này vẫn có % 'Thấp' để chạy Luật 1)
gn_sim.input['loi_nhuan'] = 50     # Trung bình

gn_sim.compute()
kq_ket_hop = gn_sim.output['ket_hop']
kq_uu_tien = gn_sim.output['uu_tien']
# 5. IN KẾT QUẢ ĐỐI CHIẾU

print("-" * 65)

print("\n[CHỈ ĐỊNH TỪ HỆ THỐNG AI]")
print(f"1. Chỉ số KẾT HỢP đơn hàng : {kq_ket_hop:.2f} %")
print(f"2. Chỉ số ƯU TIÊN giao hàng: {kq_uu_tien:.2f} %\n")

print("[ĐÁNH GIÁ KẾT QUẢ SO VỚI GIÁO TRÌNH]")
if kq_ket_hop > 50:
    print("-> Đã giải quyết được lỗi mâu thuẫn dữ liệu trong sách.")
    print("   Hệ thống quyết định KẾT HỢP NHIỀU ĐƠN (Khớp sách).")

if 30 <= kq_uu_tien <= 70:
    print("-> Quyết định mức độ ưu tiên TRUNG BÌNH (Khớp sách).")
print("=" * 65)

 HỆ THỐNG ĐIỀU PHỐI VÀ TỐI ƯU GIAO NHẬN LOGISTICS (BÀI 2.14)
-----------------------------------------------------------------

[CHỈ ĐỊNH TỪ HỆ THỐNG AI]
1. Chỉ số KẾT HỢP đơn hàng : 61.24 %
2. Chỉ số ƯU TIÊN giao hàng: 50.00 %

[ĐÁNH GIÁ KẾT QUẢ SO VỚI GIÁO TRÌNH]
-> Đã giải quyết được lỗi mâu thuẫn dữ liệu trong sách.
   Hệ thống quyết định KẾT HỢP NHIỀU ĐƠN (Khớp sách).
-> Quyết định mức độ ưu tiên TRUNG BÌNH (Khớp sách).
